# FaceGuard V2 — Fine-tuning ConvNeXt pour FER

**Roadmap 4.1, 4.2, 4.3**

Ce notebook fine-tune le modèle ConvNeXt-Base sur AffectNet avec :
- Augmentation de données standardisée (Deramgozin2023 Ch.2.4.2)
- Weighted cross-entropy pour classes déséquilibrées (Deramgozin2023 Ch.3)
- Preprocessing aligné identique à l'inférence (Jan2017 Ch.4.3.2)

**Usage** : uploader sur Kaggle avec GPU T4/P100.
Le dataset peut être chargé depuis Google Drive ou en tant que dataset Kaggle.

## 0. Téléchargement des données depuis Google Drive (optionnel)

Si tes datasets sont sur Google Drive, exécute cette section.
Sinon (dataset uploadé comme input Kaggle), passe directement à la section 1.

**Prérequis** : les dossiers doivent être en partage "Toute personne disposant du lien".

In [ ]:
# --- Installation de gdown ---
!pip install -q gdown

import gdown
import zipfile
from pathlib import Path

# ============================================================
# Liens Google Drive (fichiers partagés "Toute personne avec le lien")
# ============================================================

GDRIVE_DATASET_ID = "1-8rcmMqesnBydeWVJ30ACqWEAo8xuWmJ"   # affectnet.zip
GDRIVE_MODEL_ID   = "1r3JHCMhIryfio47lCi6yt7IpYyR7lGvJ"   # faceguard_convnext.keras

# ============================================================
# Téléchargement
# ============================================================

DOWNLOAD_DIR = Path("/kaggle/working/data")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# --- Dataset AffectNet (zip) ---
dataset_zip = DOWNLOAD_DIR / "affectnet.zip"
affectnet_dir = DOWNLOAD_DIR / "affectnet"

if not affectnet_dir.exists():
    if not dataset_zip.exists():
        print("Téléchargement d'affectnet.zip depuis Google Drive (8 GB)...")
        gdown.download(f"https://drive.google.com/uc?id={GDRIVE_DATASET_ID}",
                       str(dataset_zip), quiet=False)
    print(f"Extraction de {dataset_zip}...")
    with zipfile.ZipFile(dataset_zip, 'r') as zf:
        zf.extractall(DOWNLOAD_DIR)
    print(f"Extraction terminée dans {DOWNLOAD_DIR}")
    # Supprimer le zip pour libérer de la place
    dataset_zip.unlink()
    print(f"Zip supprimé (libération d'espace)")
else:
    print(f"Dataset déjà extrait dans {affectnet_dir}")

# --- Modèle pré-entraîné ---
model_path = DOWNLOAD_DIR / "faceguard_convnext.keras"
if not model_path.exists():
    print("\nTéléchargement du modèle ConvNeXt depuis Google Drive (1.7 GB)...")
    gdown.download(f"https://drive.google.com/uc?id={GDRIVE_MODEL_ID}",
                   str(model_path), quiet=False)
    print(f"Modèle téléchargé: {model_path}")
else:
    print(f"\nModèle déjà présent: {model_path}")

# --- Vérification ---
print("\n=== Contenu téléchargé ===")
!ls -lh {DOWNLOAD_DIR}

if affectnet_dir.exists():
    print("\n=== Structure du dataset ===")
    !ls {affectnet_dir}

    # Certains zips créent un sous-dossier supplémentaire — on cherche les vrais dossiers train/val/test
    for sub in affectnet_dir.rglob("train"):
        if sub.is_dir():
            print(f"\nDossier 'train' trouvé: {sub}")
            print(f"Classes: {sorted([p.name for p in sub.iterdir() if p.is_dir()])}")
            break

In [ ]:
import os
import numpy as np
import tensorflow as tf
from pathlib import Path

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# --- Détection automatique du dataset ---
# Le zip peut créer un sous-dossier supplémentaire → on cherche récursivement
def find_dataset_root(base: Path) -> Path | None:
    """Trouve le dossier qui contient train/, val/, test/."""
    if (base / "train").is_dir():
        return base
    for candidate in base.rglob("train"):
        if candidate.is_dir() and (candidate.parent / "val").is_dir():
            return candidate.parent
    return None

_gdrive_data = Path("/kaggle/working/data")
_kaggle_input = Path("/kaggle/input/affectnet")

DATASET_DIR = None
for candidate in [_gdrive_data, _kaggle_input]:
    if candidate.exists():
        found = find_dataset_root(candidate)
        if found:
            DATASET_DIR = found
            break

if DATASET_DIR is None:
    raise FileNotFoundError(
        "Dataset non trouvé. Soit :\n"
        "  1. Exécuter la section 0 (Google Drive)\n"
        "  2. Ajouter AffectNet comme input Kaggle"
    )

print(f"[Dataset] → {DATASET_DIR}")
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"

# --- Modèle pré-entraîné ---
_gdrive_model = _gdrive_data / "faceguard_convnext.keras"
_kaggle_model = Path("/kaggle/input/faceguard-convnext/faceguard_convnext.keras")

if _gdrive_model.exists():
    PRETRAINED_MODEL = str(_gdrive_model)
elif _kaggle_model.exists():
    PRETRAINED_MODEL = str(_kaggle_model)
else:
    raise FileNotFoundError("Modèle non trouvé.")

print(f"[Modèle] → {PRETRAINED_MODEL}")

# --- Sortie ---
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_MODEL = OUTPUT_DIR / "faceguard_convnext_finetuned.keras"
OUTPUT_TFLITE = OUTPUT_DIR / "faceguard_finetuned_fp32.tflite"

# --- Hyperparamètres ---
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
FINE_TUNE_LR = 1e-5       # LR réduit pour les couches du backbone
FINE_TUNE_AT_LAYER = 200  # Dégeler les couches du ConvNeXt à partir de cet index

EMOTION_CLASSES = ["anger", "contempt", "disgust", "fear",
                   "happy", "neutral", "sad", "surprise"]
NUM_CLASSES = len(EMOTION_CLASSES)

print(f"\n[Config] IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}")

In [ ]:
import os
import numpy as np
import tensorflow as tf
from pathlib import Path

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

# --- Chemins (adapter selon Kaggle) ---
# Sur Kaggle : /kaggle/input/affectnet/...
DATASET_DIR = Path("/kaggle/input/affectnet")
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"

# Modèle pré-entraîné (uploader comme dataset Kaggle ou depuis URL)
PRETRAINED_MODEL = "/kaggle/input/faceguard-convnext/faceguard_convnext.keras"

# Sortie
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_MODEL = OUTPUT_DIR / "faceguard_convnext_finetuned.keras"
OUTPUT_TFLITE = OUTPUT_DIR / "faceguard_finetuned_fp32.tflite"

# --- Hyperparamètres ---
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
FINE_TUNE_LR = 1e-5       # LR réduit pour les couches du backbone
FINE_TUNE_AT_LAYER = 200  # Dégeler les couches du ConvNeXt à partir de cet index

EMOTION_CLASSES = ["anger", "contempt", "disgust", "fear",
                   "happy", "neutral", "sad", "surprise"]
NUM_CLASSES = len(EMOTION_CLASSES)

## 2. Vérification du dataset

In [ ]:
from collections import Counter

def count_images(base_dir):
    counts = {}
    for cls in EMOTION_CLASSES:
        cls_dir = base_dir / cls
        if cls_dir.exists():
            n = len(list(cls_dir.glob("*")))
            counts[cls] = n
        else:
            counts[cls] = 0
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts = count_images(VAL_DIR)
test_counts = count_images(TEST_DIR)

print("=== Distribution du dataset ===")
print(f"{'Classe':<12} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 40)
for cls in EMOTION_CLASSES:
    print(f"{cls:<12} {train_counts[cls]:>8} {val_counts[cls]:>8} {test_counts[cls]:>8}")
print("-" * 40)
total_train = sum(train_counts.values())
total_val = sum(val_counts.values())
total_test = sum(test_counts.values())
print(f"{'TOTAL':<12} {total_train:>8} {total_val:>8} {total_test:>8}")

## 3. Calcul des poids de classes (Roadmap 4.2)

> **[Deramgozin2023]** Ch.3 — L'entropie croisée pondérée résout le déséquilibre
> des classes RAFdb/FER2013+ et améliore la détection de CONTEMPT et DISGUST.

In [ ]:
# Poids inversement proportionnels à la fréquence
total = sum(train_counts.values())
n_classes = len(EMOTION_CLASSES)
class_weights = {}
for i, cls in enumerate(EMOTION_CLASSES):
    count = max(train_counts[cls], 1)
    class_weights[i] = total / (n_classes * count)

print("=== Poids des classes (weighted cross-entropy) ===")
for i, cls in enumerate(EMOTION_CLASSES):
    print(f"  {cls:<12} poids={class_weights[i]:.2f}  (n={train_counts[cls]})")

## 4. Pipeline de données avec augmentation (Roadmap 4.1)

> **[Deramgozin2023]** Ch.2.4.2 — Pipeline d'augmentation validé :
> rotation ±30°, cisaillement 0.3, zoom 0.3, flip horizontal 50%
>
> **[Jan2017]** Ch.4.3.2 — Alignement cohérent train/inference

In [ ]:
import cv2

# --- CLAHE (identique à l'inférence) ---
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def apply_clahe_lab(img_bgr):
    """CLAHE sur canal L (LAB) — identique au pipeline d'inférence."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

def preprocess_image(image_path, target_size=IMG_SIZE):
    """
    Preprocessing identique à l'inférence (Roadmap 4.3) :
    - Lecture BGR
    - Resize à target_size
    - CLAHE sur L (LAB)
    - BGR → RGB
    - Normalisation [0, 1]
    """
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    img = cv2.resize(img, (target_size, target_size))
    img = apply_clahe_lab(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0

print("Fonctions de preprocessing définies.")

In [ ]:
def load_dataset(base_dir, target_size=IMG_SIZE):
    """Charge toutes les images d'un dossier structuré par classes."""
    images, labels = [], []
    for i, cls in enumerate(EMOTION_CLASSES):
        cls_dir = base_dir / cls
        if not cls_dir.exists():
            print(f"  ⚠ Classe '{cls}' non trouvée dans {base_dir}")
            continue
        files = list(cls_dir.glob("*"))
        for f in files:
            img = preprocess_image(f, target_size)
            if img is not None:
                images.append(img)
                labels.append(i)
    return np.array(images), np.array(labels)

print("Chargement du dataset d'entraînement...")
X_train, y_train = load_dataset(TRAIN_DIR)
print(f"  Train: {X_train.shape[0]} images, shape={X_train.shape[1:]}")

print("Chargement du dataset de validation...")
X_val, y_val = load_dataset(VAL_DIR)
print(f"  Val: {X_val.shape[0]} images, shape={X_val.shape[1:]}")

In [ ]:
# --- Augmentation de données (Roadmap 4.1) ---
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(30 / 360),       # ±30°
    tf.keras.layers.RandomZoom((-0.3, 0.0)),         # zoom 0-30%
    tf.keras.layers.RandomTranslation(0.1, 0.1),     # translation ±10%
    tf.keras.layers.RandomBrightness(0.2),            # luminosité ±20%
    tf.keras.layers.RandomContrast(0.2),              # contraste ±20%
], name="augmentation")

# --- Datasets tf.data ---
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(len(X_train)).batch(BATCH_SIZE)
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE,
).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Train: {len(X_train)} images en batches de {BATCH_SIZE}")
print(f"Val: {len(X_val)} images en batches de {BATCH_SIZE}")

## 5. Chargement et adaptation du modèle

In [ ]:
# Charger le modèle pré-entraîné
base_model = tf.keras.models.load_model(PRETRAINED_MODEL, compile=False)
print(f"Modèle chargé: {len(base_model.layers)} couches")
base_model.summary()

In [ ]:
# Extraire le backbone ConvNeXt (sans la couche d'augmentation du modèle original)
convnext_backbone = None
for layer in base_model.layers:
    if 'convnext' in layer.name.lower():
        convnext_backbone = layer
        break

if convnext_backbone is None:
    raise ValueError("Backbone ConvNeXt non trouvé dans le modèle")

print(f"Backbone: {convnext_backbone.name}, {len(convnext_backbone.layers)} couches")

# Phase 1 : geler le backbone
convnext_backbone.trainable = False

# Construire le nouveau modèle (sans l'augmentation du modèle original)
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = convnext_backbone(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.LayerNormalization()(x)
x = tf.keras.layers.Dense(512, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="faceguard_convnext_finetuned")
model.summary()

## 6. Phase 1 — Entraînement de la tête (backbone gelé)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5, restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7,
    ),
]

print("=== Phase 1 : Entraînement de la tête (backbone gelé) ===")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
)

## 7. Phase 2 — Fine-tuning du backbone (couches hautes dégelées)

In [ ]:
# Dégeler les dernières couches du backbone
convnext_backbone.trainable = True
for layer in convnext_backbone.layers[:FINE_TUNE_AT_LAYER]:
    layer.trainable = False

trainable = sum(1 for l in convnext_backbone.layers if l.trainable)
frozen = sum(1 for l in convnext_backbone.layers if not l.trainable)
print(f"Backbone: {trainable} couches entraînables, {frozen} gelées")

# Recompiler avec LR plus faible pour le fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

print("=== Phase 2 : Fine-tuning du backbone ===")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
)

## 8. Évaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Charger le test set
print("Chargement du test set...")
X_test, y_test = load_dataset(TEST_DIR)
print(f"Test: {X_test.shape[0]} images")

# Prédictions
y_pred_probs = model.predict(X_test, batch_size=BATCH_SIZE)
y_pred = np.argmax(y_pred_probs, axis=1)

# Rapport de classification
print("\n=== Rapport de classification ===")
print(classification_report(
    y_test, y_pred,
    target_names=[c.upper() for c in EMOTION_CLASSES],
    digits=3,
))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[c.upper() for c in EMOTION_CLASSES],
            yticklabels=[c.upper() for c in EMOTION_CLASSES])
plt.title("Matrice de confusion — ConvNeXt fine-tuné")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
# Courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Combiner les historiques
acc = history_phase1.history["accuracy"] + history_phase2.history["accuracy"]
val_acc = history_phase1.history["val_accuracy"] + history_phase2.history["val_accuracy"]
loss = history_phase1.history["loss"] + history_phase2.history["loss"]
val_loss = history_phase1.history["val_loss"] + history_phase2.history["val_loss"]
phase1_epochs = len(history_phase1.history["accuracy"])

axes[0].plot(acc, label="Train")
axes[0].plot(val_acc, label="Val")
axes[0].axvline(x=phase1_epochs, color="gray", linestyle="--", label="Fine-tune start")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(loss, label="Train")
axes[1].plot(val_loss, label="Val")
axes[1].axvline(x=phase1_epochs, color="gray", linestyle="--", label="Fine-tune start")
axes[1].set_title("Loss")
axes[1].legend()

plt.suptitle("Entraînement ConvNeXt fine-tuné (Phase 1 + Phase 2)")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "training_curves.png"), dpi=150)
plt.show()

## 9. Export du modèle

In [ ]:
# Sauvegarder le modèle Keras
model.save(str(OUTPUT_MODEL))
print(f"Modèle Keras sauvegardé: {OUTPUT_MODEL}")
print(f"Taille: {OUTPUT_MODEL.stat().st_size / 1e6:.1f} MB")

# Export TFLite FP32
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open(str(OUTPUT_TFLITE), "wb") as f:
    f.write(tflite_model)
print(f"TFLite FP32 sauvegardé: {OUTPUT_TFLITE}")
print(f"Taille: {OUTPUT_TFLITE.stat().st_size / 1e6:.1f} MB")

## 10. Vérification du modèle exporté

In [ ]:
# Vérifier que le modèle exporté donne les mêmes résultats
reloaded = tf.keras.models.load_model(str(OUTPUT_MODEL), compile=False)
sample = X_test[:5]

orig_preds = model.predict(sample, verbose=0)
reload_preds = reloaded.predict(sample, verbose=0)

print("Vérification modèle original vs rechargé:")
for i in range(5):
    orig_cls = EMOTION_CLASSES[np.argmax(orig_preds[i])]
    reload_cls = EMOTION_CLASSES[np.argmax(reload_preds[i])]
    match = "OK" if orig_cls == reload_cls else "MISMATCH"
    print(f"  Image {i}: {orig_cls} vs {reload_cls} — {match}")

print(f"\nDiff max: {np.max(np.abs(orig_preds - reload_preds)):.8f}")
print("\n=== Modèle prêt à être téléchargé et utilisé dans FaceGuard V2 ===")
print(f"Fichier: {OUTPUT_MODEL.name}")
print(f"Placer dans: models/faceguard_convnext.keras")